# 04. Core Statistical Modeling & Goodness-of-Fit Comparison
This notebook implements Maximum Likelihood Estimation (MLE) fitting of discrete (Poisson, Negative Binomial) and continuous (Normal, Exponential, Gamma, Log-Normal, Weibull) distributions. We perform Chi-Square Goodness-of-Fit (for discrete) and Kolmogorov-Smirnov (for continuous) tests, and compute AIC/BIC.


In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))
import pandas as pd
import numpy as np
from feature_engineering import aggregate_ecommerce_to_minutes, aggregate_web_server_to_minutes
from distributions import check_overdispersion, run_distribution_fitting
from visualization import plot_distribution_fits, plot_qq_plots


## Load aggregated minute data


In [ ]:
df_ec = pd.read_csv('../data/processed/clean_ecommerce_logs.csv')
df_ws = pd.read_csv('../data/processed/clean_web_server_logs.csv')
df_ec['accessed_date'] = pd.to_datetime(df_ec['accessed_date'])
df_ws['timestamp'] = pd.to_datetime(df_ws['timestamp'])

df_ec_min, _ = aggregate_ecommerce_to_minutes(df_ec)
df_ws_min, _ = aggregate_web_server_to_minutes(df_ws)


## Check for Overdispersion
We evaluate the Index of Dispersion (Variance / Mean). If Variance >> Mean, we confirm overdispersion, making standard Poisson modeling inappropriate.


In [ ]:
for label, s in [('E-Commerce All', df_ec_min['requests_per_minute']),
                 ('Web Server All', df_ws_min['requests_per_minute'])]:
    mean, var, disp, is_od = check_overdispersion(s.values)
    print(f'{label:20s}: Mean={mean:.4f}, Var={var:.4f}, Dispersion={disp:.4f} (Overdispersed: {is_od})')


## Fit Distributions to E-Commerce (Primary) Regimes
We fit candidates to all traffic and peak traffic separately to test our hypothesis of different statistical regimes.


In [ ]:
df_ec_fit_all = run_distribution_fitting(df_ec_min['requests_per_minute'].values, 'E-Commerce All')
df_ec_fit_peak = run_distribution_fitting(df_ec_min[df_ec_min['traffic_level'] == 'peak']['requests_per_minute'].values, 'E-Commerce Peak')

print('=== E-Commerce All Traffic Fits ===')
print(df_ec_fit_all[['Distribution', 'AIC', 'BIC', 'GoF Test', 'p-value']])

print('\n=== E-Commerce Peak Traffic Fits ===')
print(df_ec_fit_peak[['Distribution', 'AIC', 'BIC', 'GoF Test', 'p-value']])


## Visualizing the Fits and Q-Q Plots


In [ ]:
%matplotlib inline
plot_distribution_fits(df_ec_min, df_ec_fit_all, '../outputs/figures/ec_distribution_fits_all.png', 'E-Commerce', 'all')
plot_distribution_fits(df_ec_min, df_ec_fit_peak, '../outputs/figures/ec_distribution_fits_peak.png', 'E-Commerce', 'peak')
plot_qq_plots(df_ec_min, df_ec_fit_all, '../outputs/figures/ec_qq_plots.png', 'E-Commerce')
